In [ ]:
   !pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb pypdf sentence-transformers langgraph

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("IT-Policy.pdf")
pages = loader.load()

print("Number of pages:", len(pages))
print(pages[0].page_content[:500])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # max characters per chunk
    chunk_overlap=200    # characters repeated between neighbouring chunks
)

chunks = splitter.split_documents(pages)

print("Number of chunks:", len(chunks))
print("---- Chunk 10 ----")
print(chunks[10].page_content)
print("Metadata:", chunks[10].metadata)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test = embeddings.embed_query("What is the password policy?")
print("Length of the embedding:", len(test))
print("First 5 numbers:", test[:5])

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="it_policy",
    embedding_function=embeddings,
    persist_directory="chroma_db"
)
vectorstore.reset_collection()      # start empty on every run, so no duplicates
vectorstore.add_documents(chunks)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Stored chunks:", vectorstore._collection.count())   # should print 76

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

THRESHOLD = 1.3

class RAGState(TypedDict):
    question: str
    context: str
    pages: List[str]
    best_score: float
    answer: str
    escalated: bool

def retrieve(state: RAGState):
    results = vectorstore.similarity_search_with_score(state["question"], k=3)
    context = "\n\n".join(doc.page_content for doc, _ in results)
    pages = sorted({doc.metadata["page_label"] for doc, _ in results})
    best = min(score for _, score in results)
    return {"context": context, "pages": pages, "best_score": best}

def generate(state: RAGState):
    prompt = (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say \"I don't know\".\n\n"
        f"Question: {state['question']}\n\n"
        f"Context:\n{state['context'][:1500]}\n\n"
        "Answer:"
    )
    return {"answer": generate_text(prompt), "escalated": False}

def escalate(state: RAGState):
    return {
        "answer": "I'm not confident I can answer this from the policy document. "
                  "I have forwarded your question to a human agent.",
        "escalated": True,
    }

def route(state: RAGState):
    return "escalate" if state["best_score"] > THRESHOLD else "generate"

graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("escalate", escalate)

graph.set_entry_point("retrieve")
graph.add_conditional_edges("retrieve", route, {"generate": "generate", "escalate": "escalate"})
graph.add_edge("generate", END)
graph.add_edge("escalate", END)

app = graph.compile()

In [ ]:
def ask(question):
    result = app.invoke({"question": question})
    print("Question  :", question)
    print("Best score:", round(result["best_score"], 3))
    print("Escalated :", result["escalated"])
    print("Answer    :", result["answer"])
    print("Sources   : pages", result["pages"])
    print()

# Questions the document can answer
ask("Who does this policy apply to?")
ask("What is the password policy?")

# A question the document cannot answer: should escalate to a human
ask("What is the capital of France?")

# Known limitation: the search matches the word "leave", not its meaning
ask("What is the leave policy for employees?")